In [39]:
import numpy as np
import pandas as pd
import os

cols = ["Rndrng_NPI", "Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
        "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Stdzd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

# same cleaning as data_preprocessing.ipynb
keep = ((df["Rndrng_Prvdr_Cntry"] == "US")
        & (df["Tot_Srvcs"] >= df["Tot_Benes"])
        & (df["Tot_Bene_Day_Srvcs"] <= df["Tot_Srvcs"])
        & (df["Avg_Mdcr_Alowd_Amt"] > 0))
df = df[keep].reset_index(drop=True)
df["row_id"] = np.arange(len(df))
len(df) 

9781161

## Columns creation!

In [ ]:
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))  # payment level (avoid log(0) = -inf for $0 amounts)
df["log_benes"] = np.log(df["Tot_Benes"])   # volume
df["srvcs_per_bene"] = np.log(df["Tot_Srvcs"] / df["Tot_Benes"]) # intensity
df["markup"] = np.log(df["Avg_Sbmtd_Chrg"].clip(lower=1e-6) / df["Avg_Mdcr_Alowd_Amt"])  # charge vs allowed

feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]   

# Peer groupping

In [95]:
MIN = 30 #peer group minimum size - I did some experiments with this and 30 seems to be a good balance between having enough peer groups and having enough members in each peer group.
levels = [["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],   # level 1
          ["HCPCS_Cd", "Place_Of_Srvc"], # level 2 (fallback)
          ["HCPCS_Cd"]] #level 3 (fallback)

df["peer_level"] = 0 # 0 = unscored, 1 = level 1, 2 = level 2, 3 = level 3
df["peer_id"] = -1 # no group id yet
df["peer_n"] = 0 # group size
for i, c in enumerate(levels, start=1):
    size = df.groupby(c, observed=True)["row_id"].transform("size")
    gid = df.groupby(c, observed=True).ngroup() + i * 10_000_000   # unique group id for each level (10M apart to avoid collisions)
    take = (df["peer_level"] == 0) & (size >= MIN)
    df.loc[take, "peer_id"] = gid[take]
    df.loc[take, "peer_n"] = size[take]
    df.loc[take, "peer_level"] = i

(df["peer_level"].value_counts(normalize=True) * 100).round(2)  # 0 = unscored

peer_level
1    97.63
2     2.03
0     0.24
3     0.10
Name: proportion, dtype: float64

# Robust z scores for our 5 columns

In [96]:
s = df[df["peer_level"]>0]
med = s.groupby("peer_id")[feats].transform("median") # median of each feature within each peer group
dev = (s[feats] - med).abs() # absolute deviation from median
mad = dev.groupby(s["peer_id"]).transform("median")  # median absolute deviation
scale = (1.4826 * mad).where(mad > 0, 1.2533 * dev.groupby(s["peer_id"]).transform("mean"))
rz = (s[feats] - med) / scale.where(scale > 0) # robust z-score (NaN if scale = 0)

rz_cols = [f"{f}_rz" for f in feats]
df[rz_cols] = rz.set_axis(rz_cols, axis=1) # rename columns, rows align by index (unscored -> NaN)

In [97]:
import os
os.makedirs("data/processed", exist_ok=True)

model_input = df[["row_id", "Rndrng_NPI", "peer_level", "peer_id", "peer_n"] + rz_cols].copy()
model_input[rz_cols] = model_input[rz_cols].astype("float32")      
model_input.to_parquet("data/processed/model_input.parquet", index=False)

## Explorations

In [98]:
model_input.head()

,row_id,Rndrng_NPI,peer_level,peer_id,peer_n,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz
0,0,1003000126,1,10026929,1213,-2.119651,0.920317,0.000000,0.501598
1,1,1003000126,1,10026931,18024,-1.696201,2.022970,0.000000,0.083158
2,2,1003000126,1,10026933,32360,-0.020690,0.251892,-0.674491,0.964575
3,3,1003000126,1,10026935,5592,-18.361212,-1.035447,-0.522472,0.230335
4,4,1003000126,1,10026937,35610,-3.948478,-0.515971,-0.413015,2.400614


In [45]:
model_input[model_input["peer_level"] == 0]["Rndrng_NPI"].nunique()

14702

In [99]:
df[df["peer_level"] == 0]["Rndrng_Prvdr_Type"].value_counts().head(10)

Rndrng_Prvdr_Type
Clinical Laboratory           4272
Orthopedic Surgery            1368
Ambulatory Surgical Center    1144
Diagnostic Radiology           905
General Surgery                886
Otolaryngology                 834
Ophthalmology                  801
Physician Assistant            708
Urology                        702
Nurse Practitioner             642
Name: count, dtype: int64

In [100]:
df['high_markup'] = df['markup_rz']>3.5

In [101]:
df[["high_markup"]].value_counts(normalize=True).mul(100).round(2)

high_markup
False          98.55
True            1.45
Name: proportion, dtype: float64

In [102]:
df.groupby("Place_Of_Srvc")["high_markup"].mean().mul(100).round(2)

C:\Users\talic\AppData\Local\Temp\ipykernel_29460\1830304588.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Place_Of_Srvc")["high_markup"].mean().mul(100).round(2)


Place_Of_Srvc
F    1.32
O    1.52
Name: high_markup, dtype: float64

In [103]:
df["chrg_to_alowd"] = df["Avg_Sbmtd_Chrg"] / df["Avg_Mdcr_Alowd_Amt"]
by_type = df.groupby("Rndrng_Prvdr_Type", observed = True).agg(
    rows = ("chrg_to_alowd", "size"),
   med_markup = ("chrg_to_alowd", "median")
)
by_type[by_type['rows']>10000].sort_values("med_markup", ascending=False).head(10)

,rows,med_markup
Rndrng_Prvdr_Type,,
Anesthesiology,203587,9.193096
Certified Registered Nurse Anesthetist (CRNA),115717,8.769750
Emergency Medicine,210968,5.852103
Independent Diagnostic Testing Facility (IDTF),31764,5.792212
Ambulatory Surgical Center,60937,4.728252
Diagnostic Radiology,1119939,4.355949
Interventional Radiology,59869,4.315429
Pathology,144616,4.151911
Pain Management,44627,4.074121


In [104]:
df["prov_high_rows"]= df.groupby("Rndrng_NPI")["high_markup"].transform("sum")

In [105]:
df[df["Rndrng_NPI"] == "1992998702"]

,Rndrng_NPI,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,Rndrng_Prvdr_Cntry,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,...,high_markup,charge/allowed,chrg_to_alowd,prov_high_rows,n_flags,baseline_raw,score,reason,flag,risk_tier
9781074,1992998702,Centralized Flu,90480,O,US,144,152.0,152,41.838224,38.788487,...,True,1.078625,1.078625,2,1,554.258086,99.9,markup,False,High
9781075,1992998702,Centralized Flu,90653,O,US,214,214.0,214,83.490000,81.820000,...,False,1.020411,1.020411,2,0,1.434678,67.5,log_benes,False,None
9781076,1992998702,Centralized Flu,90677,O,US,20,20.0,20,297.102000,291.161000,...,False,1.020405,1.020405,2,0,0.803021,40.4,log_pay,False,None
9781077,1992998702,Centralized Flu,91320,O,US,71,73.0,73,151.483562,148.452603,...,False,1.020417,1.020417,2,0,0.000000,4.5,NaN,False,None
9781078,1992998702,Centralized Flu,91322,O,US,69,74.0,74,159.311757,156.127838,...,False,1.020393,1.020393,2,0,0.412278,20.6,log_pay,False,None
9781079,1992998702,Centralized Flu,G0008,O,US,218,221.0,221,41.128597,30.945656,...,True,1.329059,1.329059,2,1,60.626469,99.7,markup,False,High
9781080,1992998702,Centralized Flu,G0009,O,US,20,20.0,20,31.248500,27.225000,...,False,1.147787,1.147787,2,0,3.057465,91.3,markup,False,Low


In [106]:
results = df.groupby("Rndrng_NPI").agg(
    n_rows = ("high_markup", "size"),
    n_high = ("high_markup", "sum"),
    share_high = ("high_markup", "mean")
)
top = results[(results["n_rows"]>= 20) & (results["share_high"]>0.5)].sort_values("share_high", ascending=False)
types = df[["Rndrng_NPI", "Rndrng_Prvdr_Type"]].drop_duplicates()
top = top.merge(types, left_index=True, right_on="Rndrng_NPI")
top = top.reset_index(drop=True)

In [107]:
top

,n_rows,n_high,share_high,Rndrng_NPI,Rndrng_Prvdr_Type
0,20,19,0.950000,1053330647,General Surgery
1,23,21,0.913043,1124242029,Vascular Surgery
2,20,18,0.900000,1609971852,Diagnostic Radiology
3,24,21,0.875000,1699081117,General Surgery
4,23,20,0.869565,1699790931,Cardiology
...,...,...,...,...,...
240,45,23,0.511111,1821118811,Diagnostic Radiology
241,53,27,0.509434,1104208982,Diagnostic Radiology
242,55,28,0.509091,1053350298,Diagnostic Radiology
243,55,28,0.509091,1942232657,Diagnostic Radiology


In [108]:
eligible = results[results["n_rows"] >= 20].merge(types, left_index=True, right_on="Rndrng_NPI")
eligible["Rndrng_Prvdr_Type"].value_counts(normalize=True).head(5)   

Rndrng_Prvdr_Type
Diagnostic Radiology    0.220200
Cardiology              0.095590
Family Practice         0.089308
Internal Medicine       0.088281
Dermatology             0.041194
Name: proportion, dtype: float64

In [109]:
check = df[df["Rndrng_NPI"].isin(top["Rndrng_NPI"]) & df["high_markup"]]
check.groupby("Rndrng_Prvdr_Type")["Place_Of_Srvc"].value_counts(normalize=True).loc["Diagnostic Radiology"]

C:\Users\talic\AppData\Local\Temp\ipykernel_29460\2197345982.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  check.groupby("Rndrng_Prvdr_Type")["Place_Of_Srvc"].value_counts(normalize=True).loc["Diagnostic Radiology"]


Place_Of_Srvc
F    0.913116
O    0.086884
Name: proportion, dtype: float64

Markup is a noisy feature for some specialties (e.g. radiology), because of billing structure, not only provider behavior.
- Power BI: add a note for radiology: "markup flags may reflect billing structure".
- Models: if Isolation Forest also ranks many radiologists at the top, this explains why.

## Baseline model

In [120]:
z = df[rz_cols].clip(lower=0)              # high side only, NOT capped at 10 here
n_flags = (z > 3.5).sum(axis=1)

print(f"1+ feature > 3.5:            {(n_flags >= 1).mean():.2%}")
print(f"2+ features > 3.5:           {(n_flags >= 2).mean():.2%}")
print(f"one > 10 OR 2+ > 3.5:        {((z.max(axis=1) > 10) | (n_flags >= 2)).mean():.2%}")

1+ feature > 3.5:            5.86%
2+ features > 3.5:           0.16%
one > 10 OR 2+ > 3.5:        1.19%


In [124]:
high = df[rz_cols].clip(lower=0)  # only above peers
scored = df["peer_level"] > 0

z_sorted = np.sort(high.fillna(0).to_numpy(), axis=1) # each row sorted ascending
top1, top2 = z_sorted[:, -1], z_sorted[:, -2]# largest and 2nd largest z

# anomaly: one feature > 10 OR two+ features > 3.5  <=>  baseline_raw > 3.5
df["baseline_raw"] = pd.Series(np.maximum(top2, top1 * 3.5 / 10), index=df.index).where(scored)

df["n_flags"] = (high > 3.5).sum(axis=1)
df["flag"] = df["baseline_raw"] > 3.5
df["score"] = (df["baseline_raw"].rank(pct=True) * 100).round(1)
df["risk_tier"] = "None"
fl = df["flag"]
df.loc[fl, "risk_tier"] = pd.cut(df.loc[fl, "baseline_raw"].rank(pct=True),
                                 [0, 0.5, 0.85, 1], labels=["Low", "Medium", "High"]).astype(str)
names = [c.replace("_rz", "") for c in rz_cols]
mask = (high > 3.5).to_numpy()
df["reason"] = [", ".join(n for n, m in zip(names, row) if m) for row in mask]

print(f"flagged: {df['flag'].mean():.2%}")                     
print(pd.crosstab(df["risk_tier"], df["flag"]))

flagged: 1.19%
flag         False  True 
risk_tier                
High             0  17398
Low              0  57992
Medium           0  40595
None       9665176      0


In [125]:
out = df[["row_id", "score", "flag", "reason", "risk_tier"]].copy()
out.insert(1, "method", "baseline_zscore")
out.to_parquet("data/processed/baseline_output.parquet", index=False)
out.head()

,row_id,method,score,flag,reason,risk_tier
0,0,baseline_zscore,53.8,False,,None
1,1,baseline_zscore,69.2,False,,None
2,2,baseline_zscore,39.0,False,,None
3,3,baseline_zscore,14.3,False,,None
4,4,baseline_zscore,76.3,False,,None


In [126]:
out.sort_values("score", ascending=False).head(10)

,row_id,method,score,flag,reason,risk_tier
3297693,3297693,baseline_zscore,100.0,True,markup,High
3297694,3297694,baseline_zscore,100.0,True,markup,High
3298090,3298090,baseline_zscore,100.0,True,markup,High
8251281,8251281,baseline_zscore,100.0,True,markup,High
1137242,1137242,baseline_zscore,100.0,True,markup,High
792111,792111,baseline_zscore,100.0,True,markup,High
791940,791940,baseline_zscore,100.0,True,markup,High
7254184,7254184,baseline_zscore,100.0,True,markup,High
5258170,5258170,baseline_zscore,100.0,True,markup,High
6589253,6589253,baseline_zscore,100.0,True,markup,High


In [118]:
df.sort_values("baseline_raw", ascending=False)[["row_id", "baseline_raw", "reason"] + rz_cols].head(10)

,row_id,baseline_raw,reason,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz
6326823,6326823,5.850140e+10,markup,0.0,-0.774811,-1.254000,5.850140e+10
640831,640831,4.996891e+10,markup,0.0,0.471130,2.618855,4.996891e+10
2698091,2698091,4.758287e+10,markup,0.0,0.529332,-0.369106,4.758287e+10
8258214,8258214,4.758287e+10,markup,0.0,1.508386,0.545367,4.758287e+10
7919249,7919249,4.758287e+10,markup,0.0,0.021483,0.253557,4.758287e+10
1703718,1703718,4.758287e+10,markup,0.0,1.163443,-0.423368,4.758287e+10
3389394,3389394,4.758287e+10,markup,0.0,1.127883,2.253458,4.758287e+10
1714115,1714115,4.758287e+10,markup,0.0,1.258588,1.089370,4.758287e+10
2491600,2491600,4.716190e+10,markup,0.0,0.357469,0.766171,4.716190e+10
1612516,1612516,4.625463e+10,markup,0.0,1.433740,-0.252622,4.625463e+10
